# AI Solution Architect Agent (Terminal Demo)
Thin command-line demo. All logic lives in **`architect.py`**
(Single Source of Truth) – the same file that `app.py` (Streamlit UI)
uses. Changes therefore only need to be made in `architect.py`.

**Prerequisite:** `./chroma_db` exists (built once via `Rag_Setup.ipynb`).

## 1) SETUP – load model & database from architect.py

In [ ]:
from architect import get_model, get_db, send_message

model = get_model()
conn = get_db()

count = conn.execute('SELECT COUNT(*) FROM conversations').fetchone()[0]
print("Setup complete. Model ready.")
print(f"Saved messages in DB: {count}")

## 2) RAG Quick Test – query the knowledge base
Checks whether the Chroma vector store is reachable and returns hits.

In [ ]:
from architect import search_patterns

print(search_patterns("Microservices")[:300] + "...")

## 3) CHAT LOOP – interactive conversation with the agent

Type your message and press Enter. Use `quit` to end the chat.

In [ ]:
print("="*60)
print("AI Solution Architect – chat started")
print("Type 'quit' to exit, 'history' for chat history")
print("="*60)

while True:
    user_input = input("\nYou: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() == "quit":
        print("\nChat ended. Goodbye!")
        break
    
    if user_input.lower() == "history":
        print("\n--- Chat History ---")
        for mid, role, content, ts in conn.execute(
            "SELECT id, role, content, timestamp FROM conversations ORDER BY id ASC"
        ).fetchall():
            label = "You" if role == "user" else "Architect"
            print(f"[{ts[:19]}] {label}: {content[:100]}...")
        print("--- End ---")
        continue
    
    try:
        answer, in_tok, out_tok = send_message(conn, model, user_input)
        print(f"\n[Tokens: input={in_tok}, output={out_tok}]")
        print(f"\nArchitect: {answer}")
    except Exception as e:
        print(f"\nError: {e}")

conn.close()
print("Database connection closed.")